In [ ]:
import os, re, difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ruptures as rpt
from statsmodels.tsa.filters.hp_filter import hpfilter
from sklearn.preprocessing import StandardScaler

# 1. 路径及参数
base = r"D:\研\大论文\数据\热度数据"
hourly = os.path.join(base, "分年份数据")
influence = os.path.join(base, "影响因素分析_213起事件综合表.xlsx")
out = os.path.join(base, "HP生命周期分析_HP100")
os.makedirs(out, exist_ok=True)

LAMBDA = 100
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.unicode_minus": False
})

# 2. 读取事件名单及匹配小时数据
targets = pd.read_excel(influence)
event_col = next(c for c in targets.columns if "事件" in str(c) or "名称" in str(c))
year_col = next(c for c in targets.columns if "年份" in str(c) or str(c).lower() == "year")
targets = targets.dropna(subset=[event_col]).drop_duplicates(event_col)

files = []
for year in range(2021, 2027):
    folder = os.path.join(hourly, f"单事件小时数据{year}")
    if os.path.exists(folder):
        files += [(str(year), f) for f in os.listdir(folder) if f.endswith(".csv")]

def clean(s):
    s = os.path.splitext(os.path.basename(str(s)))[0].lower()
    s = re.sub(r"^20\d{2}", "", s)
    return re.sub(r"[^\u4e00-\u9fa50-9]", "", s)

def match(name, year):
    key = clean(name)
    candidates = [(y, f) for y, f in files if y == year] if year else files
    if not candidates:
        candidates = files
    scores = [(difflib.SequenceMatcher(None, key, clean(f)).ratio(), y, f)
              for y, f in candidates]
    if not scores:
        return None
    score, y, f = max(scores)
    return (y, os.path.join(hourly, f"单事件小时数据{y}", f)) if score >= 0.85 else None

# 3. Fisher最优分段及生命周期划分
def lifecycle(y, trend):
    n = len(y)
    z = StandardScaler().fit_transform(y.reshape(-1, 1)).ravel()
    algo = rpt.Dynp(model="l2", jump=1, min_size=max(3, n // 20)).fit(z)
    max_k = min(5, n // 10 + 1)

    costs = []
    for k in range(1, max_k + 1):
        start, cost = 0, 0
        for b in algo.predict(n_bkps=k):
            seg = z[start:b]
            cost += np.sum((seg - seg.mean()) ** 2)
            start = b
        costs.append(cost)

    k = max(1, min(np.argmax(np.diff(costs, 2)) + 2, max_k)) if len(costs) >= 3 else 1
    bkps = sorted(b for b in algo.predict(n_bkps=k) if 0 < b < n)

    bounds = [0] + bkps + [n]
    means = [trend[bounds[i]:bounds[i+1]].mean()
             for i in range(len(bounds)-1)]

    m = len(means)
    if m == 1:
        return ["成熟期"], [0, n]

    peaks = [i for i in range(1, m-1)
             if means[i] >= means[i-1] and means[i] >= means[i+1]]

    if means[0] >= means[1]:
        peaks.insert(0, 0)
    if means[-1] >= means[-2]:
        peaks.append(m-1)

    peaks = sorted(set(peaks)) or [int(np.argmax(means))]
    peak_value = max(means[i] for i in peaks)
    peaks = [i for i in peaks if means[i] >= peak_value * 0.60]
    first, last = peaks[0], peaks[-1]

    peak_idx = np.searchsorted(bounds, np.argmax(trend), side="right") - 1

    if peak_idx == 0 or first == 0:
        for i in range(1, m):
            if means[i] < peak_value * 0.65 and all(
                v < peak_value * 0.50 for v in means[i:]
            ):
                return ["成熟期", "衰退期"], [0, bounds[i], n]
        return ["成熟期"], [0, n]

    for i in range(last + 1, m):
        tail = means[i:i+2]
        if len(tail) == 2 and means[i] < peak_value * 0.65 and all(
            v < peak_value * 0.50 for v in tail
        ):
            return ["爆发期", "成熟期", "衰退期"], [
                0, bounds[first], bounds[i], n
            ]

    return ["爆发期", "成熟期"], [0, bounds[first], n]

# 4. 绘制HP趋势及生命周期图
def plot_event(name, year, y, trend, stages, bounds):
    x = np.arange(len(y))
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(x, y, color="steelblue", alpha=0.4,
            lw=0.9, label="Hourly Posts")
    ax.plot(x, trend, color="red", lw=2,
            label=f"HP Trend (Lambda={LAMBDA})")

    colors = {"爆发期": "#DCEEDC", "成熟期": "#DCE7F5",
              "衰退期": "#F5E5D8"}
    labels = {"爆发期": "Burst Period", "成熟期": "Mature Period",
              "衰退期": "Decline Period"}

    for i, stage in enumerate(stages):
        left, right = bounds[i], bounds[i+1]
        ax.axvspan(left, right, color=colors[stage], alpha=0.45)
        ax.text((left+right)/2, 0.95, labels[stage],
                transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=11)
        if i > 0:
            ax.axvline(left, color="black", ls="--", lw=1)

    ax.set_xlabel("Time (Hours)")
    ax.set_ylabel("Hourly Weibo Posts")
    ax.set_title(f"{year} - {name}")
    ax.legend()
    ax.grid(alpha=0.2)
    ax.set_xlim(0, len(y)-1)
    ax.set_xticks(np.arange(0, len(y), 24 if len(y) <= 280 else 48))
    plt.tight_layout()

    safe = re.sub(r'[\\/:*?"<>|]', "_", name)
    plt.savefig(os.path.join(out, f"{year}_{safe}.png"),
                dpi=300, bbox_inches="tight")
    plt.close()

# 5. 批量处理213起事件
success = 0
failed = []

for _, row in targets.iterrows():
    name = str(row[event_col])
    year = str(int(float(row[year_col]))) if pd.notna(row[year_col]) else None

    info = match(name, year)
    if info is None:
        failed.append(name)
        continue

    year, file = info

    try:
        df = None
        for enc in ["utf-8-sig", "gb18030", "gbk"]:
            try:
                df = pd.read_csv(file, encoding=enc)
                break
            except UnicodeDecodeError:
                continue

        if df is None or "发文量" not in df.columns:
            raise ValueError("无法读取文件或缺少发文量列")

        y = pd.to_numeric(df["发文量"], errors="coerce").fillna(0).to_numpy(float)
        if len(y) < 10:
            raise ValueError("数据点少于10")

        _, trend = hpfilter(y, lamb=LAMBDA)
        trend = np.maximum(np.asarray(trend), 0)

        stages, bounds = lifecycle(y, trend)
        plot_event(name, year, y, trend, stages, bounds)

        success += 1
        print(f"完成：{year} {name}")

    except Exception as e:
        failed.append(name)
        print(f"失败：{name} | {e}")

print(f"\n处理完成：成功 {success} 起，失败 {len(failed)} 起")
print(f"图片保存位置：{out}")
if failed:
    print("未成功处理事件：", failed)